# Table Models for Observability Data

**Module 4: Table Models for Observability Data** · [Lesson](course.md) · [Quiz](quiz04_observability_table_models.ipynb)

Since Module 1, about half of the store's orders fail. During an incident and after it, people ask the same questions again and again: how many requests each service handled in each minute and how many failed, and which kinds of errors occurred. In this Lab you keep the answers in compact tables, and see how the model of a table decides what Doris does with rows that have the same key:

| Model | Rows with the same key | Use in observability |
| --- | --- | --- |
| `DUPLICATE KEY` | All are kept. The key columns only set the sort order. | Raw logs, spans, and metric points |
| `AGGREGATE KEY` | Combined into one row. Each value column names its aggregate function, such as `SUM`. | Counts kept ahead of time, such as requests and failures per service and minute |
| `UNIQUE KEY` | Only the row loaded last is kept. | Current state, such as one alert per kind of error and day |

You check which models the exporter chose for the telemetry tables, build an AGGREGATE table of the requests and failed requests of each service and minute, and build a UNIQUE table that keeps one alert per kind of error and day. The examples read the spans in `otel_traces`. The new tables are created in the `otel` database, and every cell can be rerun.

Plan about 35 minutes. You need the DOG stack and the demo from Module 1, with the payment failure turned on (section 5 of Lab 1).

### Initialize the Lab

Run this cell once before using `lab.shell(...)` or `lab.sql(...)`.

In [ ]:
from pathlib import Path
import sys

COURSE_ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "doris_course").is_dir()
)
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from doris_course import DorisLab

lab = DorisLab(lab_dir=COURSE_ROOT)

## 1. Check the failed orders

The examples summarize the spans in `otel_traces`, among them the spans of the failed orders. Count the orders since the demo started, and how many failed:

In [ ]:
orders = lab.wait_for_rows(
    """
    SELECT COUNT(*) AS orders,
           SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_orders,
           MIN(timestamp) AS first_order,
           MAX(timestamp) AS latest_order
    FROM otel_traces
    WHERE service_name = 'checkout'
      AND span_name = 'oteldemo.CheckoutService/PlaceOrder'
    HAVING SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) > 0
    """,
    title="Orders and failed orders in otel_traces",
    hint="No order has failed. Check that the demo runs (lab.start_demo() in Module 1) and that paymentFailure "
         "is 50% (section 5 of Lab 1), wait a minute or two, then rerun this cell.",
    timeout_seconds=180,
)

**Expected result**

One row: the orders since the demo started, usually a few dozen to a few hundred, the failed orders among them, about half of those placed since you turned on the flag, and the times of the first and the latest order in UTC. While only a few orders have been placed since the flag, chance can put the share of failures far from half.

## 2. DUPLICATE: keep every event

The Doris exporter created the telemetry tables when it started in Module 1. `information_schema.columns` marks each key column with its table's model: `DUP` for DUPLICATE, `AGG` for AGGREGATE, and `UNI` for UNIQUE. List the models and key columns of these tables:

In [ ]:
lab.sql("""
SELECT TABLE_NAME AS table_name,
       MAX(COLUMN_KEY) AS model,
       GROUP_CONCAT(COLUMN_NAME ORDER BY ORDINAL_POSITION SEPARATOR ', ') AS key_columns
FROM information_schema.columns
WHERE TABLE_SCHEMA = 'otel' AND TABLE_NAME LIKE 'otel%' AND COLUMN_KEY <> ''
GROUP BY TABLE_NAME
ORDER BY table_name
""", title="Models and key columns of the telemetry tables");

**Expected result**

Eight tables. Seven are `DUP`: `otel_logs`, `otel_traces`, and the five `otel_metrics_*` tables, keyed on `timestamp` and `service_name` (in either order). `otel_traces_graph` is `UNI`; section 4 explains why.

A log line, a span, and a metric point are each a separate event, so the raw tables use DUPLICATE: Doris keeps every row it loads. The key columns only set the sort order, which helps queries that filter on time and service. DUPLICATE is also the model that `CREATE TABLE` uses when it has no key clause.

The key of `otel_traces` is `service_name` and `timestamp`, the time at which the span started. Look for keys that several spans share:

In [ ]:
lab.sql("""
SELECT service_name, timestamp,
       COUNT(*) AS spans_with_this_key,
       COUNT(DISTINCT span_id) AS different_spans,
       COUNT(DISTINCT span_name) AS different_operations
FROM otel_traces
GROUP BY service_name, timestamp
HAVING COUNT(*) > 1
ORDER BY spans_with_this_key DESC, timestamp
LIMIT 5
""", title="Keys that several spans in otel_traces share");

**Expected result**

Five keys, each shared by about ten spans, usually of `frontend` or `frontend-web`, the storefront's code on the server and in the browser. Their SDKs record times to the millisecond, and one request starts several spans in the same millisecond. `different_spans` equals `spans_with_this_key`: every span has its own span ID, so these rows are different events, and a table that kept one row per key would lose all but one of them. DUPLICATE keeps exact copies too: it cannot protect a table from loading the same data twice.

## 3. AGGREGATE: count requests and failures per service and minute

A dashboard of the incident shows, for each service and minute, how many requests the service handled and how many of them failed. Computed from `otel_traces` on every refresh, it reads every span of the period. An AGGREGATE table keeps the counts instead, one row per service and minute.

A request that a service handles is a server span, whose `span_kind` is `SPAN_KIND_SERVER`. For `checkout`, these are the orders, and for `payment`, the charges. In the next table, the key columns are the reporting grain: `request_minute` and `service_name`. The value columns `requests` and `failed_requests` declare `SUM`: when a row arrives whose key the table already has, Doris adds its values to the existing row. Doris combines such rows while it loads them, merges the rest in the background (compaction), and combines anything left at query time, so a query always sees one row per key.

The key holds the minute, not the time of each span. With a span's time in the key, only spans from the same microsecond would be combined, and the table would be almost as large as `otel_traces`. Each request counts in the minute in which it ended, `DATE_TRUNC(end_time, 'minute')`: a span reaches Doris only after it ends, so a long request, such as one of the flag streams of `flagd`, which stay open for up to 10 minutes, would arrive long after the minute in which it started had been counted. `DISTRIBUTED BY HASH(service_name)` sends all rows of a service to the same tablet; one bucket is enough for a table this small. Module 5 explains how to choose partitions and buckets.

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS requests_per_minute (
    request_minute  DATETIME,
    service_name    VARCHAR(200),
    requests        BIGINT SUM,
    failed_requests BIGINT SUM
)
AGGREGATE KEY(request_minute, service_name)
DISTRIBUTED BY HASH(service_name) BUCKETS 1
""", title="Create the AGGREGATE table requests_per_minute");

**Expected result**

A card says that the statement succeeded. `IF NOT EXISTS` leaves an existing table unchanged, so you can rerun the cell.

### Load the spans in two batches

A load job usually runs on a schedule and adds the rows of the period that has just ended. The next cell loads the server spans of `otel_traces` into `requests_per_minute` in two such batches: the second batch is the spans that ended in a period of 30 seconds, and the first batch is every span that ended before. Spans reach Doris a few seconds after they end, so the cell leaves out the last 30 seconds before the newest span.

Each span becomes one row with `requests = 1`, and `failed_requests = 1` when its status is `STATUS_CODE_ERROR`, `0` otherwise. After each batch, the cell shows the table's rows from the minute in which the second batch starts. Before you run the cell, predict: does the second batch add rows for that minute, or change the rows that are there?

In [ ]:
bounds = lab.sql("""
SELECT MAX(end_time) - INTERVAL 60 SECOND AS second_batch_start,
       MAX(end_time) - INTERVAL 30 SECOND AS second_batch_end,
       DATE_TRUNC(MAX(end_time) - INTERVAL 60 SECOND, 'minute') AS boundary_minute
FROM otel_traces
""", title="Time range of the second batch")
second_batch_start = bounds["second_batch_start"][0]
second_batch_end = bounds["second_batch_end"][0]
boundary_minute = bounds["boundary_minute"][0]
lab.execute("TRUNCATE TABLE requests_per_minute", title="Empty the table, so that the cell can be rerun")

LOAD_REQUESTS = """
INSERT INTO requests_per_minute (request_minute, service_name, requests, failed_requests)
SELECT DATE_TRUNC(end_time, 'minute'), service_name, 1, IF(status_code = 'STATUS_CODE_ERROR', 1, 0)
FROM otel_traces
WHERE span_kind = 'SPAN_KIND_SERVER' AND {period}
"""
BOUNDARY_ROWS = f"""
SELECT * FROM requests_per_minute
WHERE request_minute >= '{boundary_minute}'
ORDER BY request_minute, requests DESC
"""

lab.execute(LOAD_REQUESTS.format(period=f"end_time < '{second_batch_start}'"),
            title="First batch: the server spans that ended before the second batch")
lab.sql(BOUNDARY_ROWS, title="requests_per_minute from the minute in which the second batch starts, after the first batch")

lab.execute(LOAD_REQUESTS.format(period=f"end_time >= '{second_batch_start}' AND end_time < '{second_batch_end}'"),
            title="Second batch: the server spans that ended in 30 seconds")
lab.sql(BOUNDARY_ROWS, title="The same rows after the second batch");

**Expected result**

First the period of the second batch, 30 seconds that end 30 seconds before the newest span, and the minute in which it starts. The first batch reports one affected row for each span that it loaded, tens of thousands, yet `requests_per_minute` has only one row per service and minute. After the first batch, the rows of the boundary minute hold the requests of its first seconds, those before `second_batch_start`.

The second batch loads a few hundred to a few thousand spans. Afterwards, the boundary minute still has one row per service: a service that handled requests in both batches, such as `frontend-proxy` or `frontend`, keeps its row, and its `requests` has grown by its spans in the second batch. This is the `SUM` merge: loads of 4 and then 7 for one key leave one row with 11. A service whose first request of the minute came in the second batch gets a new row, and so does every service of the next minute when the second batch reaches into it.

An AGGREGATE table cannot tell new rows from a repeated load. Had the second batch been loaded twice, its spans would have been added twice, and the counts would be too high. A load job must load every source row exactly once; this cell starts with `TRUNCATE TABLE`, so you can rerun it.

Check the counts against the raw spans. The next query counts the server spans of `otel_traces` per service and minute, up to the end of the second batch, and compares each count with the row of `requests_per_minute`:

In [ ]:
lab.sql(f"""
SELECT COUNT(*) AS groups_in_otel_traces,
       SUM(IF(s.requests = r.requests AND s.failed_requests = r.failed_requests, 1, 0)) AS groups_that_match,
       (SELECT COUNT(*) FROM requests_per_minute) AS rows_in_requests_per_minute,
       SUM(r.requests) AS server_spans_read
FROM (
    SELECT DATE_TRUNC(end_time, 'minute') AS request_minute, service_name,
           COUNT(*) AS requests,
           SUM(IF(status_code = 'STATUS_CODE_ERROR', 1, 0)) AS failed_requests
    FROM otel_traces
    WHERE span_kind = 'SPAN_KIND_SERVER' AND end_time < '{second_batch_end}'
    GROUP BY DATE_TRUNC(end_time, 'minute'), service_name
) AS r
LEFT JOIN requests_per_minute AS s
  ON s.request_minute = r.request_minute AND s.service_name = r.service_name
""", title="requests_per_minute checked against the server spans in otel_traces");

**Expected result**

One row, with the same number in the first three columns: one group for each service and minute, every group matches its row in `requests_per_minute`, and the table has no other rows. `server_spans_read` is the number of spans that this query read to compute the counts, tens of thousands; `requests_per_minute` gives the same answer from one row per service and minute. The price is that the table answers only questions at its grain: it cannot say which requests failed, or how long they took.

If `groups_that_match` is smaller, some spans of that period reached Doris after the cell above loaded it: rerun both cells.

Now read the incident from the summary: the orders, `checkout`, and the charges, `payment`, of each minute, with their failures.

In [ ]:
lab.sql("""
SELECT request_minute, service_name, requests, failed_requests,
       ROUND(100 * failed_requests / requests) AS failed_percent
FROM requests_per_minute
WHERE service_name IN ('checkout', 'payment')
ORDER BY request_minute DESC, service_name
LIMIT 20
""", title="Orders and charges per minute, newest first");

**Expected result**

Up to 20 rows, two for most minutes: `checkout` counts the orders, one to four a minute, and `payment` the charges. Every order makes one charge, so the two counts are usually equal; they differ by one when an order ended a moment after its charge, in the next minute. Since you turned on the flag, about half of the charges fail, and with them their orders: the same failures, seen in two services. With so few orders, a single minute can have none or all of them failed.

## 4. UNIQUE: one alert per kind of error and day

Suppose that an alerting job reads the error spans every minute and raises an alert for each. Every failed order leaves error spans in several services, so one alert per error span is an alert storm: several alerts a minute, all about the same problem. A UNIQUE table keeps one row per key, so with the day and an alert identity as the key, each kind of error raises one alert per day, however often it occurs.

The alert identity must be the same for every occurrence of a kind of error and differ between kinds. Here it is the service, the operation (`span_name`), and the status message of the span. Messages can contain parts that change from one occurrence to the next, such as addresses, ports, and IDs, so the query below replaces every number in the message with `0`, and hashes the result together with the service and the operation: `alert_hash`. See how it groups the error spans:

In [ ]:
alerts = lab.wait_for_rows(
    r"""
    SELECT DATE(timestamp) AS alert_date,
           service_name,
           span_name,
           MD5(CONCAT_WS('|', service_name, span_name, REGEXP_REPLACE(status_message, '[0-9]+', '0'))) AS alert_hash,
           COUNT(*) AS error_spans,
           MAX(end_time) AS last_error,
           LEFT(MIN(REGEXP_REPLACE(status_message, '[0-9]+', '0')), 80) AS normalized_message
    FROM otel_traces
    WHERE status_code = 'STATUS_CODE_ERROR'
    GROUP BY alert_date, service_name, span_name, alert_hash
    ORDER BY error_spans DESC, alert_hash
    """,
    title="Error spans by alert identity",
    hint="otel_traces has no error spans. Turn on the payment failure with section 5 of Lab 1, wait for "
         "the first failed order, then rerun this cell.",
    timeout_seconds=10,
)

**Expected result**

One row per alert and day, usually 15 to 20. The kinds of error of the incident come first, each with one error span per failed order: the payment service's `grpc.oteldemo.PaymentService/Charge`, the checkout service's `oteldemo.CheckoutService/PlaceOrder` and its call to payment, `oteldemo.PaymentService/Charge`, and the spans of `frontend`, `frontend-proxy`, and `load-generator` that carried the failed request. Below them come kinds with one error span each, from the minute in which the demo started, such as `payment` and `frontend` failing to reach `metadata.google.internal`, a cloud metadata service, and `accounting` reading a Kafka topic that did not exist yet. `normalized_message` shows why numbers are replaced: `recommendation`'s stream of flag updates, `/flagd.evaluation.v1.Service/EventStream`, ends with `DEADLINE_EXCEEDED` every 10 minutes, with a message that contains the time; with the numbers replaced, every occurrence gets the same identity. That row appears once the demo has run for 10 minutes. `last_error` is the time at which the last of an alert's spans ended.

Create the alert table. Its key is the day and the alert identity:

In [ ]:
lab.execute("""
CREATE TABLE IF NOT EXISTS error_alerts (
    alert_date   DATE,
    alert_hash   VARCHAR(32),
    service_name VARCHAR(200),
    span_name    STRING,
    status       VARCHAR(16),
    message      STRING
)
UNIQUE KEY(alert_date, alert_hash)
DISTRIBUTED BY HASH(alert_hash) BUCKETS 1
""", title="Create the UNIQUE table error_alerts");

**Expected result**

A card says that the statement succeeded. A UNIQUE table must be distributed by a hash of key columns, so that all rows with one key reach the same tablet. Doris 4.x creates UNIQUE tables as Merge-on-Write (MoW) by default: when a row arrives whose key the table already has, Doris marks the old row as deleted while it loads the new one, so queries read only the newest row.

Raise an open alert for every error span, as the alerting job would:

In [ ]:
RAISE_ALERTS = r"""
INSERT INTO error_alerts (alert_date, alert_hash, service_name, span_name, status, message)
SELECT DATE(timestamp),
       MD5(CONCAT_WS('|', service_name, span_name, REGEXP_REPLACE(status_message, '[0-9]+', '0'))),
       service_name,
       span_name,
       'open',
       status_message
FROM otel_traces
WHERE status_code = 'STATUS_CODE_ERROR'
"""
lab.execute("TRUNCATE TABLE error_alerts", title="Empty the table, so that the cell can be rerun")
lab.execute(RAISE_ALERTS, title="Raise an open alert for every error span")
lab.sql("SELECT * FROM error_alerts ORDER BY service_name, span_name", title="error_alerts");

**Expected result**

The insert reports one affected row per error span, nine for each failed order and a few others, yet `error_alerts` has one row per alert and day, as many as the previous query returned: rows with the same key replaced each other. Each row keeps the message of one occurrence, the one that Doris loaded last.

One row does not mean one error. The alert of the payment service's charge has one row in `error_alerts` and one error span per failed order in `otel_traces`. The table keeps the current state, not a count: to know how often an error occurred, count its spans in `otel_traces`, as the previous query did, or keep a count in an AGGREGATE table, as section 3 does.

A minute later, the alerting job runs again and reads the same error spans. Load them once more:

In [ ]:
lab.execute(RAISE_ALERTS, title="Raise the alerts again from the same error spans")
lab.sql("SELECT COUNT(*) AS alerts FROM error_alerts", title="Alerts after the second load");

**Expected result**

The insert loads the same number of rows again, plus the error spans of any order that failed in between, and `alerts` has not changed. Repeated loads replace rows instead of adding them, so a UNIQUE table can be loaded again safely. A repeated load of `requests_per_minute` in section 3 would have counted every span twice.

When a problem is fixed, the alerting job writes its alert again, with the status `resolved`. The payment failure goes on, but the errors of the demo's start happened once. Resolve the alert whose last error is the oldest:

In [ ]:
oldest = alerts.sort_values("last_error").iloc[0]
print(f"Resolve the alert of {oldest['service_name']} {oldest['span_name']}, "
      f"last error at {oldest['last_error']} UTC")
lab.execute(f"""
INSERT INTO error_alerts (alert_date, alert_hash, service_name, span_name, status, message)
SELECT alert_date, alert_hash, service_name, span_name, 'resolved', message
FROM error_alerts
WHERE alert_date = '{oldest['alert_date']}' AND alert_hash = '{oldest['alert_hash']}'
""", title="Write the alert again with the status resolved")
lab.sql(f"SELECT * FROM error_alerts WHERE alert_hash = '{oldest['alert_hash']}'", title="The resolved alert")
lab.sql("SELECT status, COUNT(*) AS alerts FROM error_alerts GROUP BY status ORDER BY status", title="Alerts by status");

**Expected result**

First the alert that the cell resolves, usually one of the demo's start. The insert writes one row, and the alert still has one row, now with `status = resolved`. `Alerts by status` adds up to the same number of alerts as before, one of them resolved: a later load with the same key replaced the open row.

If you now rerun the cell that raises the alerts again, this alert is `open` again, because that load is later still: UNIQUE keeps the row loaded last. When loads can arrive out of order, give the table a sequence column (the `function_column.sequence_col` property): Doris then keeps the row with the largest value in that column, such as the time of the alert's last error, whatever the load order. A resolved row written with the time of the resolution then stays until an error newer than that time is loaded.

### The exporter's UNIQUE table

`otel_traces_graph`, the UNIQUE table from section 2, counts the calls between services per minute. A job that the exporter created, `otel_traces_graph_job`, fills it every 10 minutes: each run counts the spans in `otel_traces` from the start of the minute 10 minutes earlier up to the current time. Each run therefore counts again the minute in which the previous run started, which the previous run saw only in part, and writes rows with the same keys. UNIQUE replaces the earlier, partial counts with the new ones. With AGGREGATE `SUM`, the two counts would be added; with DUPLICATE, the minute would appear twice.

Look at the job with `jobs('type'='insert')`, which lists the scheduled insert jobs, and at the calls from `frontend-proxy` to `frontend` in the newest minutes:

In [ ]:
graph_job = lab.sql("""
SELECT Name, RecurringStrategy, Status, SucceedTaskCount, LastTaskSuccessTime
FROM jobs('type'='insert')
WHERE Name LIKE '%otel_traces_graph_job'
""", title="The job that fills otel_traces_graph")
lab.sql("""
SELECT timestamp, caller_service_name, callee_service_name, count, error_count
FROM otel_traces_graph
WHERE caller_service_name = 'frontend-proxy' AND callee_service_name = 'frontend'
ORDER BY timestamp DESC
LIMIT 5
""", title="Calls from frontend-proxy to frontend per minute");

**Expected result**

The job runs `EVERY 10 MINUTE`, and `SucceedTaskCount` says how often it has run. Most minutes have a similar number of calls, about 1,450 while the demo runs. The newest minutes can have fewer calls, or no row yet: the last run counted them before all of their spans had reached Doris. The next two cells keep one minute fixed, so that it can still be found after it has left the newest five rows.

`error_count` counts the calls that failed. The incident shows in the calls from `checkout` to `payment`: change the two service names in the second query and rerun it, and about half of the calls of each minute since the flag was turned on have failed.

If `SucceedTaskCount` is 0 and the second table is empty, the job has not run yet. The DOG Collector creates the schema with `create_schema: true`: restarting it drops and recreates the job, whose first run is 10 minutes later. Module 2's trace collector has `create_schema: false`; restarting that collector does not recreate the job. If the first run has not finished, rerun the job check above and the capture cell below after it finishes.

Run this cell **once after a successful job run** to remember the minute of `LastTaskSuccessTime`. It prints the target and its current count. Keep the demo running. After the next scheduled run, rerun only the comparison cell below, so that the remembered minute stays the same. After a kernel restart, capture a new baseline first.

In [ ]:
from datetime import datetime

if int(graph_job["SucceedTaskCount"].iloc[0]) > 0:
    graph_minute = datetime.fromisoformat(str(graph_job["LastTaskSuccessTime"].iloc[0])).replace(second=0, microsecond=0)
    GRAPH_MINUTE_QUERY = f"""
    SELECT timestamp, caller_service_name, callee_service_name, count, error_count
    FROM otel_traces_graph
    WHERE timestamp = '{graph_minute:%Y-%m-%d %H:%M:%S}'
      AND caller_service_name = 'frontend-proxy' AND callee_service_name = 'frontend'
    """
    graph_before = lab.sql(GRAPH_MINUTE_QUERY, title=f"Baseline for the fixed minute {graph_minute} UTC")
else:
    graph_minute = None
    print("The graph job has not completed a run. Wait for its first run, then rerun the job check and this capture cell.")

After the next job run, execute this comparison cell again. The job check shows when the latest run finished; the second query always selects the captured minute, even when it is more than five rows back. Expect the same service pair in one row with its count filled in; the count cannot decrease while the same input spans are retained.

In [ ]:
if graph_minute is None:
    print("Capture a baseline after the graph job's first successful run before comparing it.")
else:
    lab.sql("""
    SELECT Name, SucceedTaskCount, LastTaskSuccessTime
    FROM jobs('type'='insert')
    WHERE Name LIKE '%otel_traces_graph_job'
    """, title="Check whether the next scheduled graph run has finished")
    lab.sql(GRAPH_MINUTE_QUERY, title=f"Compare the same minute {graph_minute} UTC with the baseline above");

## 5. Exercise: a daily summary of the logs

Build the same kind of summary for the logs: `daily_log_counts`, with the key `log_date` and `service_name`, and the `SUM` columns `log_count` and `error_count`. A log row is an error when its `severity_number` is 17 or more, the ERROR and FATAL range of OpenTelemetry. Load it from `otel_logs` up to a fixed time, 30 seconds before the newest log row, then compare it with a `GROUP BY` query on `otel_logs` with the same time limit. Expect one row per service and day, and the same numbers in both results. Which services have error rows, and why does the payment failure add none?

In [ ]:
# Your code here

The next cell is a reference solution; it is collapsed. Expand it to compare, or run it.

In [ ]:
log_limit = lab.sql(
    "SELECT MAX(timestamp) - INTERVAL 30 SECOND AS log_limit FROM otel_logs",
    title="Load the log rows before this time",
)["log_limit"][0]

lab.execute("""
CREATE TABLE IF NOT EXISTS daily_log_counts (
    log_date     DATE,
    service_name VARCHAR(200),
    log_count    BIGINT SUM,
    error_count  BIGINT SUM
)
AGGREGATE KEY(log_date, service_name)
DISTRIBUTED BY HASH(service_name) BUCKETS 1
""", title="Create the AGGREGATE table daily_log_counts")
lab.execute("TRUNCATE TABLE daily_log_counts", title="Empty the table, so that the cell can be rerun")
lab.execute(f"""
INSERT INTO daily_log_counts (log_date, service_name, log_count, error_count)
SELECT DATE(timestamp), service_name, 1, IF(severity_number >= 17, 1, 0)
FROM otel_logs
WHERE timestamp < '{log_limit}'
""", title="Load the log rows")
lab.sql("SELECT * FROM daily_log_counts ORDER BY log_count DESC", title="daily_log_counts")
lab.sql(f"""
SELECT DATE(timestamp) AS log_date, service_name,
       COUNT(*) AS log_count,
       SUM(IF(severity_number >= 17, 1, 0)) AS error_count
FROM otel_logs
WHERE timestamp < '{log_limit}'
GROUP BY DATE(timestamp), service_name
ORDER BY log_count DESC
""", title="The same counts computed from otel_logs");

**Expected result**

The same rows and numbers in both tables, about 11 services: `frontend-proxy` and `doris-fe`, the FE log of Module 2, have the most rows. Few services have error rows: `doris-fe`, from the errors that the FE logged when it started, and sometimes a demo service such as `accounting`. The payment failure adds none: the payment and checkout services send no logs, and the proxy's access log line of a failed checkout has no severity, as Module 2 showed. A summary of error logs would miss this incident; the summary of section 3, built from spans, shows it.

## You are done when

- you can say which model the exporter uses for raw telemetry, and why spans that share a key in `otel_traces` must all be kept;
- `requests_per_minute` has one row per service and minute, the second batch added to the existing rows, and the counts match the server spans in `otel_traces`;
- `error_alerts` has one row per alert and day, however often its error occurred and however often the alerts were loaded, and the alert that you resolved shows `resolved`;
- you can explain why `otel_traces_graph` is a UNIQUE table.

The Lab's tables are small, and the later modules do not use them. To remove them, run `DROP TABLE IF EXISTS requests_per_minute`, `DROP TABLE IF EXISTS error_alerts`, and `DROP TABLE IF EXISTS daily_log_counts` with `lab.execute(...)` or in a MySQL client.

## Next

Open [Quiz 4: Table Models for Observability Data](quiz04_observability_table_models.ipynb), then continue to [Module 5: Partitioning and Bucketing for Observability Data](../module05-partition-bucket/course.md).